# **Preprocessing v2**

**Problem** dataset v1 (`dataset_final_model.csv`) stuck di R² 0.67:

1. Fitur cuaca (prediktif) dibuang di preprocessing v1. ex: `cloudcover`, `winddir`, `tempmax`, `feelslike`, `uvindex` punya korelasi |r| ≥ 0.23 dengan PM2.5 tapi tidak dimasukkan ke dataset final
2. 2094 dari 8776 (24%) day station tidak punya data per jam. Jadi aggregate ke daily mean → NaN → diisi forward/backward fill atau median

**Perbaikan di v2:**
- Tambah fitur cuaca: `cloudcover`, `tempmax`, `tempmin`, `feelslike`, `dew`, `uvindex`, `precipprob`, `sealevelpressure`
- Encode `winddir` sebagai `winddir_sin` & `winddir_cos` (variabel circular: 0° = 360°)
- Filter quality: drop day-station dengan observasi jam-an PM2.5 < 12 (separuh hari)
- Drop sensor error: PM2.5 = 0
- Tidak menggunakan ffill/bfill/median di target PM2.5, kalau ada data baris dibuang

In [1]:
import pandas as pd

df_v2 = pd.read_csv("../data/final for modelling/dataset_final_model_v2.csv")
df_v2.head()

,tanggal,ISPU PM2.5,kategori_pm25,bulan,hari_minggu,temp,humidity,visibility,windgust,solarenergy,...,pm25_std,pm25_max,n_obs,station_bundaran hi,station_jagakarsa,station_kebun jeruk,station_kelapa gading,station_lubang buaya,station_us embassy 1,station_us embassy 2
0,2022-01-13,49.916667,Baik,1,3,27.1,85.4,6.2,27.4,11.3,...,0.996205,51.0,12,1,0,0,0,0,0,0
1,2022-01-14,46.916667,Baik,1,4,26.8,86.9,5.7,28.8,19.4,...,4.440687,51.0,12,1,0,0,0,0,0,0
2,2022-01-15,52.750000,Sedang,1,5,26.9,87.1,5.8,30.2,14.5,...,0.452267,53.0,12,1,0,0,0,0,0,0
3,2022-01-18,51.066667,Sedang,1,1,26.1,91.9,5.1,40.7,13.8,...,14.169720,57.0,15,1,0,0,0,0,0,0
4,2022-01-19,39.687500,Baik,1,2,27.0,85.5,5.9,55.4,20.2,...,12.185887,52.0,16,1,0,0,0,0,0,0


In [2]:
fitur_cuaca_cek = [
    'temp',
    'tempmax',
    'tempmin',
    'feelslike',
    'dew',
    'humidity',
    'precip',
    'precipprob',
    'windgust',
    'windspeed',
    'winddir_sin',
    'winddir_cos',
    'sealevelpressure',
    'cloudcover',
    'visibility',
    'solarenergy',
    'uvindex'
]

target = 'ISPU PM2.5'

fitur_ada = [kolom for kolom in fitur_cuaca_cek if kolom in df_v2.columns]

korelasi_pm25 = (
    df_v2[fitur_ada + [target]]
    .corr(numeric_only=True)[target]
    .drop(target)
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

tabel_korelasi = korelasi_pm25.reset_index()
tabel_korelasi.columns = ['fitur', 'r']
tabel_korelasi['abs_r'] = tabel_korelasi['r'].abs()
tabel_korelasi['masuk_abs_r_023'] = tabel_korelasi['abs_r'] >= 0.23

tabel_korelasi

,fitur,r,abs_r,masuk_abs_r_023
0,winddir_sin,0.406370,0.406370,True
1,cloudcover,-0.348539,0.348539,True
2,temp,0.331255,0.331255,True
3,tempmax,0.315411,0.315411,True
4,solarenergy,0.310086,0.310086,True
5,humidity,-0.302920,0.302920,True
6,winddir_cos,0.280035,0.280035,True
7,windgust,-0.262131,0.262131,True
8,uvindex,0.258863,0.258863,True
9,feelslike,0.256527,0.256527,True


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DIR_DATA = Path('../data')
PATH_POLUSI = DIR_DATA / 'merged data polusi' / 'polusi_merged.csv'
PATH_CUACA  = DIR_DATA / 'data cuaca' / 'cuaca_merged.csv'
PATH_OUT    = DIR_DATA / 'final for modelling' / 'dataset_final_model_v2.csv'

TARGET = 'ISPU PM2.5'
MIN_OBS_PER_HARI = 12  # threshold observasi jam minimal per hari stasiun

print('Polusi:', PATH_POLUSI)
print('Cuaca :', PATH_CUACA)
print('Out   :', PATH_OUT)

Polusi: ..\data\merged data polusi\polusi_merged.csv
Cuaca : ..\data\data cuaca\cuaca_merged.csv
Out   : ..\data\final for modelling\dataset_final_model_v2.csv


## Load Polusi (Hourly)

Data polusi mentah berformat jam dengan `"-"` sebagai marker missing (string). Konversi ke numerik dulu sebelum operasi numerik apapun

In [5]:
df_polusi = pd.read_csv(PATH_POLUSI)
print('Shape polusi:', df_polusi.shape)
print('Stasiun raw:', df_polusi['station'].unique())

# Bersihkan nama stasiun
df_polusi['station'] = df_polusi['station'].str.replace(' CSV', '', regex=False).str.strip().str.lower()

# Parse datetime
df_polusi['datetime'] = pd.to_datetime(
    df_polusi['tanggal'].astype(str) + ' ' + df_polusi['Waktu'].astype(str),
    format='%Y-%m-%d %H:%M', errors='coerce'
)
df_polusi['tanggal'] = pd.to_datetime(df_polusi['tanggal'])

# Konversi "-" ke NaN untuk kolom ISPU
KOLOM_ISPU = ['ISPU PM10', 'ISPU PM2.5', 'ISPU SO2', 'ISPU CO', 'ISPU O3', 'ISPU NO2']
for c in KOLOM_ISPU:
    if c in df_polusi.columns:
        df_polusi[c] = pd.to_numeric(df_polusi[c], errors='coerce')

print()
print('Setelah konversi numerik:')
print(f'  Total baris jam-an : {len(df_polusi):,}')
print(f'  NaN {TARGET}       : {df_polusi[TARGET].isna().sum():,} ({df_polusi[TARGET].isna().mean()*100:.1f}%)')
print(f'  Non-NaN            : {df_polusi[TARGET].notna().sum():,}')
print(f'  Range non-NaN      : {df_polusi[TARGET].min():.1f} - {df_polusi[TARGET].max():.1f}')

Shape polusi: (210624, 17)
Stasiun raw: <StringArray>
[  'Bundaran HI CSV',     'Jagakarsa CSV',   'Jakarta GBK CSV',
   'Kebun Jeruk CSV', 'Kelapa Gading CSV',  'Lubang Buaya CSV',
  'US Embassy 1 CSV',  'US Embassy 2 CSV']
Length: 8, dtype: str

Setelah konversi numerik:
  Total baris jam-an : 210,624
  NaN ISPU PM2.5       : 82,891 (39.4%)
  Non-NaN            : 127,733
  Range non-NaN      : 0.0 - 281.0


## Quality Filter: Minimal 12 Observasi Jam per Hari Stasiun

Threshold 12/24 (separuh hari) memastikan minimal ada sample yang cukup dari berbagai jam

In [ ]:
# Hitung observasi jam-an per (stasiun, tanggal)
obs_per_hari = (
    df_polusi.groupby(['station', 'tanggal'])[TARGET]
    .agg(['count', 'mean', 'std', 'min', 'max'])
    .reset_index()
)
obs_per_hari.columns = ['station', 'tanggal', 'n_obs', 'pm25_mean', 'pm25_std', 'pm25_min', 'pm25_max']

# Statistik
print(f'Total day-station combinations  : {len(obs_per_hari):,}')
print(f'Day-station dengan 0 observasi   : {(obs_per_hari["n_obs"] == 0).sum():,}')
print(f'Day-station dengan <12 observasi : {(obs_per_hari["n_obs"] < MIN_OBS_PER_HARI).sum():,}')
print(f'Median observasi per hari        : {obs_per_hari["n_obs"].median():.0f} dari 24 jam')

# Filter ke day station yang memenuhi threshold
obs_lulus = obs_per_hari[obs_per_hari['n_obs'] >= MIN_OBS_PER_HARI].copy()
print(f'\nSetelah filter (n_obs >= {MIN_OBS_PER_HARI}): {len(obs_lulus):,} day-station')

# Stasiun yang lulus
print('\nDistribusi per stasiun:')
print(obs_lulus['station'].value_counts())

Total day-station combinations  : 8,776
Day-station dengan 0 observasi   : 2,094
Day-station dengan <12 observasi : 2,686
Median observasi per hari        : 19 dari 24 jam

Setelah filter (n_obs >= 12): 6,090 day-station

Distribusi per stasiun:
station
us embassy 1     979
us embassy 2     818
bundaran hi      802
jagakarsa        802
kebun jeruk      802
kelapa gading    802
lubang buaya     802
jakarta gbk      283
Name: count, dtype: int64


## Drop Sensor Error PM2.5 = 0

Empat baris dengan `pm25_mean == 0` di stasiun kebun jeruk awal 2023. Investigasi: cuaca normal (humidity 80-87%, temperature 26-28°C, sedikit hujan) PM2.5 = 0 di Jakarta tidak masuk akal. Pasti sensor mati

In [7]:
sensor_zero = obs_lulus[obs_lulus['pm25_mean'] == 0]
print(f'Baris PM2.5 mean = 0: {len(sensor_zero)}')
if len(sensor_zero) > 0:
    print(sensor_zero[['station', 'tanggal', 'n_obs', 'pm25_min', 'pm25_max']].to_string(index=False))

# Drop
obs_clean = obs_lulus[obs_lulus['pm25_mean'] > 0].copy()
print(f'\nSetelah drop sensor error: {len(obs_clean):,} day-station')

Baris PM2.5 mean = 0: 191
      station    tanggal  n_obs  pm25_min  pm25_max
  bundaran hi 2024-01-29     16       0.0       0.0
  bundaran hi 2024-10-19     18       0.0       0.0
  bundaran hi 2024-10-23     18       0.0       0.0
  bundaran hi 2024-10-26     12       0.0       0.0
    jagakarsa 2022-03-15     19       0.0       0.0
    jagakarsa 2022-03-24     20       0.0       0.0
    jagakarsa 2022-06-21     22       0.0       0.0
    jagakarsa 2022-08-01     19       0.0       0.0
    jagakarsa 2023-02-08     16       0.0       0.0
    jagakarsa 2024-01-29     16       0.0       0.0
    jagakarsa 2024-09-11     17       0.0       0.0
    jagakarsa 2024-11-28     18       0.0       0.0
  kebun jeruk 2022-02-17     14       0.0       0.0
  kebun jeruk 2022-03-24     20       0.0       0.0
  kebun jeruk 2022-10-06     17       0.0       0.0
  kebun jeruk 2022-10-17     21       0.0       0.0
  kebun jeruk 2022-10-18     19       0.0       0.0
  kebun jeruk 2022-10-19     20       

## Aggregate Polusi Harian (Mean) + Statistik Intraday

Selain mean, `pm25_std` dan `pm25_max` sebagai fitur tambahan untuk model: variabilitas dalam-hari (`std`) dan puncak harian (`max`) bisa lebih prediktif daripada mean saja

In [8]:
df_polusi_daily = obs_clean[['station', 'tanggal', 'pm25_mean', 'pm25_std', 'pm25_max', 'n_obs']].rename(
    columns={'pm25_mean': TARGET}
)
df_polusi_daily['pm25_std'] = df_polusi_daily['pm25_std'].fillna(0)  # std NaN ketika n_obs=1
print(f'Polusi daily shape: {df_polusi_daily.shape}')
print(df_polusi_daily.describe().round(2))

Polusi daily shape: (5899, 6)
                          tanggal  ISPU PM2.5  pm25_std  pm25_max    n_obs
count                        5899     5899.00   5899.00   5899.00  5899.00
mean   2023-06-16 07:34:31.876589       73.81      5.07     80.62    20.26
min           2022-01-07 00:00:00        2.55      0.00      4.14    12.00
25%           2022-08-07 00:00:00       59.72      1.69     66.00    18.00
50%           2023-07-22 00:00:00       74.95      2.92     81.00    20.00
75%           2024-04-01 00:00:00       88.17      5.03     94.57    24.00
max           2025-01-01 00:00:00      223.94     73.37    281.00    24.00
std                           NaN       21.92      7.48     22.33     3.32


## Load Cuaca + Tambah Fitur yang Sebelumnya Dibuang

Dari analisis korelasi, fitur ini punya |r| > 0.18 dengan PM2.5: `cloudcover`, `winddir`, `tempmax`, `feelslike`, `uvindex`, `precipprob`, `tempmin`, `sealevelpressure`. Semua ambil di v2

In [9]:
df_cuaca = pd.read_csv(PATH_CUACA)
df_cuaca['datetime'] = pd.to_datetime(df_cuaca['datetime'])
df_cuaca['tanggal'] = df_cuaca['datetime']
df_cuaca['station'] = df_cuaca['station'].str.strip().str.lower()

# Fitur cuaca yang dipakai di v2
KOLOM_CUACA_V2 = [
    
    # Dipakai di v1
    'temp', 'humidity', 'visibility', 'windgust', 'solarenergy', 'precip',
    
    # Tambahan v2 (korelasi |r| > 0.18)
    'cloudcover',         # r = -0.30
    'tempmax',            # r = +0.30
    'winddir',            # r = -0.30 (akan diencode circular)
    'feelslike',          # r = +0.26
    'uvindex',            # r = +0.23
    'precipprob',         # r = -0.18
    'tempmin',            # r = +0.12
    'sealevelpressure',   # r = +0.12 (potensial penanda tekanan synoptic)
]
print('Fitur cuaca v2:', KOLOM_CUACA_V2)
print('Jumlah fitur cuaca:', len(KOLOM_CUACA_V2))

# Konversi numerik
for c in KOLOM_CUACA_V2:
    if c in df_cuaca.columns:
        df_cuaca[c] = pd.to_numeric(df_cuaca[c], errors='coerce')

# Cek availability
tersedia = [c for c in KOLOM_CUACA_V2 if c in df_cuaca.columns]
tidak_ada = [c for c in KOLOM_CUACA_V2 if c not in df_cuaca.columns]
print(f'Tersedia: {len(tersedia)} | Tidak ada: {tidak_ada}')
KOLOM_CUACA_V2 = tersedia

Fitur cuaca v2: ['temp', 'humidity', 'visibility', 'windgust', 'solarenergy', 'precip', 'cloudcover', 'tempmax', 'winddir', 'feelslike', 'uvindex', 'precipprob', 'tempmin', 'sealevelpressure']
Jumlah fitur cuaca: 14
Tersedia: 14 | Tidak ada: []


## Encode `winddir` sebagai Circular (sin/cos)

Arah angin adalah variabel circular: 0° = 360°. Kalau dipakai sebagai numerik mentah, model akan melihat 1° dan 359° sebagai "berjauhan" padahal sebenarnya berdekatan. Encode dengan sin/cos memetakan ke circle

In [10]:
if 'winddir' in df_cuaca.columns:
    df_cuaca['winddir_sin'] = np.sin(2 * np.pi * df_cuaca['winddir'] / 360)
    df_cuaca['winddir_cos'] = np.cos(2 * np.pi * df_cuaca['winddir'] / 360)
    # Ganti winddir mentah dengan sin/cos di daftar fitur
    KOLOM_CUACA_V2 = [c for c in KOLOM_CUACA_V2 if c != 'winddir'] + ['winddir_sin', 'winddir_cos']
    print('winddir di-encode circular -> winddir_sin, winddir_cos')

print('\nFinal fitur cuaca v2:', KOLOM_CUACA_V2)

winddir di-encode circular -> winddir_sin, winddir_cos

Final fitur cuaca v2: ['temp', 'humidity', 'visibility', 'windgust', 'solarenergy', 'precip', 'cloudcover', 'tempmax', 'feelslike', 'uvindex', 'precipprob', 'tempmin', 'sealevelpressure', 'winddir_sin', 'winddir_cos']


## Merge Polusi Daily + Cuaca

Inner join pada (station, tanggal). Drop jakarta_gbk untuk konsistensi dengan v1.

In [11]:
df_merged = df_polusi_daily.merge(
    df_cuaca[['station', 'tanggal'] + KOLOM_CUACA_V2],
    on=['station', 'tanggal'], how='inner'
)
print(f'Setelah merge: {df_merged.shape}')

# Drop jakarta_gbk (konsisten dengan v1)
df_merged = df_merged[df_merged['station'] != 'jakarta gbk'].reset_index(drop=True)
print(f'Setelah drop jakarta_gbk: {df_merged.shape}')

# Cek NaN pada fitur cuaca
print('\nNaN per fitur cuaca:')
print(df_merged[KOLOM_CUACA_V2].isna().sum())

Setelah merge: (5892, 21)
Setelah drop jakarta_gbk: (5609, 21)

NaN per fitur cuaca:
temp                0
humidity            0
visibility          0
windgust            0
solarenergy         0
precip              0
cloudcover          0
tempmax             0
feelslike           0
uvindex             0
precipprob          0
tempmin             0
sealevelpressure    0
winddir_sin         0
winddir_cos         0
dtype: int64


## Feature Waktu + Kategori ISPU + One-Hot Station

In [12]:
df_final = df_merged.copy()
df_final['bulan'] = df_final['tanggal'].dt.month
df_final['hari_minggu'] = df_final['tanggal'].dt.dayofweek

def kategori_ispu(v):
    if v <= 50: return 'Baik'
    if v <= 100: return 'Sedang'
    if v < 200: return 'Tidak Sehat'
    if v < 300: return 'Sangat Tidak Sehat'
    return 'Berbahaya'
df_final['kategori_pm25'] = df_final[TARGET].apply(kategori_ispu)

# One-hot station
df_final = pd.get_dummies(df_final, columns=['station'], prefix='station', drop_first=False)
kolom_stasiun = [c for c in df_final.columns if c.startswith('station_')]
df_final[kolom_stasiun] = df_final[kolom_stasiun].astype(int)

# Susun kolom akhir
kolom_final = (
    ['tanggal', TARGET, 'kategori_pm25', 'bulan', 'hari_minggu']
    + KOLOM_CUACA_V2
    + ['pm25_std', 'pm25_max', 'n_obs']   # statistik intraday + quality marker
    + kolom_stasiun
)
df_final = df_final[kolom_final]
print(f'Shape final: {df_final.shape}')
print(f'Kolom: {df_final.columns.tolist()}')
print(f'\nDistribusi kategori_pm25:')
print(df_final['kategori_pm25'].value_counts())

Shape final: (5609, 30)
Kolom: ['tanggal', 'ISPU PM2.5', 'kategori_pm25', 'bulan', 'hari_minggu', 'temp', 'humidity', 'visibility', 'windgust', 'solarenergy', 'precip', 'cloudcover', 'tempmax', 'feelslike', 'uvindex', 'precipprob', 'tempmin', 'sealevelpressure', 'winddir_sin', 'winddir_cos', 'pm25_std', 'pm25_max', 'n_obs', 'station_bundaran hi', 'station_jagakarsa', 'station_kebun jeruk', 'station_kelapa gading', 'station_lubang buaya', 'station_us embassy 1', 'station_us embassy 2']

Distribusi kategori_pm25:
kategori_pm25
Sedang                4326
Baik                   726
Tidak Sehat            554
Sangat Tidak Sehat       3
Name: count, dtype: int64


## Save

In [15]:
PATH_OUT.parent.mkdir(exist_ok=True, parents=True)
df_final.to_csv(PATH_OUT, index=False)
print(f'Saved: {PATH_OUT}')
print(f'Shape: {df_final.shape}')

# Perbandingan dengan v1
import os
v1_path = DIR_DATA / 'final for modelling' / 'dataset_final_model.csv'
if v1_path.exists():
    df_v1 = pd.read_csv(v1_path)
    print(f'\nPerbandingan v1 vs v2')
    print(f'v1 shape: {df_v1.shape}')
    print(f'v2 shape: {df_final.shape}')
    print(f'v1 kolom cuaca: temp, humidity, visibility, windgust, solarenergy, precip ({6})')
    print(f'v2 kolom cuaca: + cloudcover, tempmax, tempmin, feelslike, uvindex, precipprob, winddir_sin, winddir_cos, sealevelpressure ({len(KOLOM_CUACA_V2)})')
    print(f'v1 target PM2.5 = 0: {(df_v1[TARGET] == 0).sum()}')
    print(f'v2 target PM2.5 = 0: {(df_final[TARGET] == 0).sum()} (dibersihkan)')
    print(f'v1 baris diasumsikan banyak dari imputasi (24% day-station 0 obs)')
    print(f'v2 baris semua punya >= 12 observasi jam-an asli')

Saved: ..\data\final for modelling\dataset_final_model_v2.csv
Shape: (5609, 30)

Perbandingan v1 vs v2
v1 shape: (7679, 18)
v2 shape: (5609, 30)
v1 kolom cuaca: temp, humidity, visibility, windgust, solarenergy, precip (6)
v2 kolom cuaca: + cloudcover, tempmax, tempmin, feelslike, uvindex, precipprob, winddir_sin, winddir_cos, sealevelpressure (15)
v1 target PM2.5 = 0: 4
v2 target PM2.5 = 0: 0 (dibersihkan)
v1 baris diasumsikan banyak dari imputasi (24% day-station 0 obs)
v2 baris semua punya >= 12 observasi jam-an asli


## 11. Catatan Penting untuk Modelling

Setelah dataset v2 disimpan, modelling baru (mis. `02_Modelling_XGBoost_v3.ipynb`) harus:

1. **Load** `dataset_final_model_v2.csv` (bukan v1).
2. **Tambahkan** fitur baru ke daftar `KOLOM_NUMERIK`: `cloudcover`, `tempmax`, `tempmin`, `feelslike`, `uvindex`, `precipprob`, `sealevelpressure`, `winddir_sin`, `winddir_cos`.
3. **Pertimbangkan** memakai `pm25_std` dan `pm25_max` sebagai fitur tambahan (variabilitas intraday & puncak hari sebelumnya bisa jadi sinyal lonjakan).
4. **Tetap pakai** convention yang sama: time split 70/15/15, val_RMSE selection, retrain on train+val.
5. **Jangan menimpa** output v1/v2 — pakai suffix `_v3` untuk hasil dari dataset bersih.